In [0]:
# Databricks Notebook

# -------------------------------
# Import required libraries
# -------------------------------
from pyspark.sql import SparkSession
import logging

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

# -------------------------------
# Define paths
# Replace with DBFS or mounted path
# -------------------------------
input_file_path = "/dbfs/FileStore/test_customer_data/customers-10000.csv"
destination_table_path = "/dbfs/FileStore/tables/customer/sample_customer"

# -------------------------------
# Read CSV file
# -------------------------------
data_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "false") \
    .option("delimiter", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .csv(input_file_path)

# Display Data (Databricks way)
display(data_df)

# Create temp view
data_df.createOrReplaceTempView("sample_cust")

# -------------------------------
# Run SQL
# -------------------------------
spark.sql("""
SELECT * 
FROM sample_cust
LIMIT 10
""").show()

# -------------------------------
# Sanitize column names
# -------------------------------
data_df = data_df.toDF(*[col.replace(" ", "_") for col in data_df.columns])

# -------------------------------
# Write as Delta table
# -------------------------------
data_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(destination_table_path)

# -------------------------------
# OPTIONAL: Register as table
# -------------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS sample_customer
USING DELTA
LOCATION '{destination_table_path}'
""")

logger.info("Data successfully written to Delta table.")